# PyCaret Redux — Classification Demo

This notebook demonstrates the full classification workflow using `pycaret_redux`.

## 1. Setup — Load Data & Initialize Experiment

In [ ]:
from sklearn.datasets import load_breast_cancer
import pandas as pd

# Load the breast cancer dataset (binary classification)
data = load_breast_cancer(as_frame=True)
df = data.frame
df["target"] = data.target

print(f"Dataset shape: {df.shape}")
print(f"Target distribution:\n{df['target'].value_counts()}")
df.head()

In [ ]:
from pycaret_redux import ClassificationExperiment

exp = ClassificationExperiment()
exp.setup(
    data=df,
    target="target",
    train_size=0.8,
    normalize=True,
    normalize_method="zscore",
    session_id=42,
    fold=5,
)

## 2. Available Models & Metrics

In [ ]:
exp.models()

In [ ]:
exp.get_metrics()

## 3. Compare Models

Train and compare multiple classifiers using cross-validation.

In [ ]:
best = exp.compare_models(
    include=["lr", "dt", "rf", "nb", "knn", "ada", "gbc", "et", "lda", "ridge"],
    sort="Accuracy",
    n_select=1,
)

## 4. Create Individual Models

In [ ]:
lr = exp.create_model("lr")
print(f"\nModel type: {type(lr).__name__}")

In [ ]:
rf = exp.create_model("rf")
print(f"\nModel type: {type(rf).__name__}")

## 5. Tune Model

Hyperparameter tuning via randomized search.

In [ ]:
tuned_rf = exp.tune_model(rf, n_iter=20, optimize="F1")
print(f"\nTuned model type: {type(tuned_rf).__name__}")

## 6. Ensemble Methods

### Blending (Voting Classifier)

In [ ]:
blended = exp.blend_models([lr, rf], method="soft")

### Stacking Classifier

In [ ]:
stacked = exp.stack_models([lr, rf])

### Bagging Ensemble

In [ ]:
bagged = exp.ensemble_model(rf, n_estimators=10)

## 7. Model Plots

### ROC/AUC Curve

In [ ]:
%matplotlib inline
exp.plot_model(tuned_rf, plot="auc");

### Confusion Matrix

In [ ]:
exp.plot_model(tuned_rf, plot="confusion_matrix");

### Precision-Recall Curve

In [ ]:
exp.plot_model(tuned_rf, plot="pr");

### Feature Importance

In [ ]:
exp.plot_model(tuned_rf, plot="feature");

### Classification Report Heatmap

In [ ]:
exp.plot_model(tuned_rf, plot="class_report");

### Threshold vs Metrics (Binary Only)

In [ ]:
exp.plot_model(tuned_rf, plot="threshold");

### Calibration Curve

In [ ]:
exp.plot_model(tuned_rf, plot="calibration");

### Lift, Gain, and KS Statistic Charts

In [ ]:
exp.plot_model(tuned_rf, plot="lift");

In [ ]:
exp.plot_model(tuned_rf, plot="gain");

In [ ]:
exp.plot_model(tuned_rf, plot="ks");

## 8. Model Evaluation

In [ ]:
exp.evaluate_model(tuned_rf)

## 9. Predictions

In [ ]:
# Predict on test set
preds = exp.predict_model(tuned_rf, verbose=False)
preds.head(10)

In [ ]:
# Predict with raw probability scores
preds_raw = exp.predict_model(tuned_rf, raw_score=True, verbose=False)
preds_raw[["prediction_label", "prediction_score_0", "prediction_score_1"]].head(10)

## 10. Calibrate Model & Optimize Threshold

In [ ]:
# Calibrate probabilities
calibrated = exp.calibrate_model(tuned_rf, method="sigmoid")

In [ ]:
# Find the optimal decision threshold for F1
model, best_threshold = exp.optimize_threshold(tuned_rf, optimize="F1")
print(f"Optimal threshold: {best_threshold}")

In [ ]:
# Predict with the optimized threshold
preds_optimized = exp.predict_model(tuned_rf, probability_threshold=best_threshold, verbose=False)
print(f"Predictions with threshold={best_threshold}:")
preds_optimized["prediction_label"].value_counts()

## 11. Finalize, Save & Load Model

In [ ]:
# Finalize: retrain on full dataset (train + test)
final_model = exp.finalize_model(tuned_rf)
print(f"Finalized model: {type(final_model).__name__}")

In [ ]:
# Save model
exp.save_model(final_model, "breast_cancer_rf")

In [ ]:
# Load model back
loaded_model = exp.load_model("breast_cancer_rf")
print(f"Loaded model: {type(loaded_model).__name__}")

## 12. Custom Metrics

In [ ]:
from sklearn.metrics import balanced_accuracy_score

# Add a custom metric
exp.add_metric(
    id="bal_acc",
    name="Balanced Accuracy",
    score_func=balanced_accuracy_score,
    display_name="Bal. Acc.",
)

# See it in the metrics table
exp.get_metrics()

In [ ]:
# Now create_model will include the custom metric in CV results
gbc = exp.create_model("gbc")

In [ ]:
# Remove it when done
exp.remove_metric("bal_acc")
print("Custom metric removed.")

## 13. Multiclass Classification

Let's also test with the full iris dataset (3 classes).

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True).frame

exp_multi = ClassificationExperiment()
exp_multi.setup(data=iris, target="target", session_id=123, fold=5)

In [ ]:
best_multi = exp_multi.compare_models(include=["lr", "dt", "rf", "nb", "knn"])

In [ ]:
exp_multi.plot_model(best_multi, plot="confusion_matrix");

In [ ]:
preds_multi = exp_multi.predict_model(best_multi, verbose=False)
preds_multi[["prediction_label", "prediction_score"]].head(10)

## 14. Cleanup

In [ ]:
import os
# Clean up saved model file
if os.path.exists("breast_cancer_rf.joblib"):
    os.remove("breast_cancer_rf.joblib")
    print("Cleaned up saved model file.")

print("\nAll features tested successfully!")